# S3 event STAR — IS diagnosis (sealed OOS off-limits)

Failure forensics on the **frozen event** STAR using **research IS only**.

The event sleeve is **always Forex** (G10 + calendar). It does not follow core `UNIVERSE_STAR`.

**Do not** load sealed OOS here. Disposition: kill / paper / new H — never re-tune on the sealed slice.


## 0. Imports & Config


In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s3_trend.report import fold_val_metrics, load_star_stack
from backtest.s3_trend.research import (
    RESEARCH_IS_END_S3,
    config_from_stack,
    n_trials_ledger_total,
    star_stack_path,
)
from backtest.s3_trend.runner import run_s3_backtest
from backtest.s3_trend.tearsheet import (
    baseline_configs_from_star,
    check_fill_timing,
    cost_stress_sharpes,
    daily_regime_performance,
    early_late_is_decay,
    equity_from_returns,
    ew_long_g10_returns,
    oos_headline_metrics,
    pair_pnl_attribution,
    pair_scorecard,
    plot_drawdown,
    plot_equity_net_gross,
    spy_daily_returns,
    utilization_stats,
)
from backtest.s3_trend.walkforward import embargo_bars_for_config, make_s3_folds
from data.processing.s3_fx_price_panel import price_panel_path, s3_data_dir
from strategies.s3_trend.metrics import (
    load_s1_period_returns,
    load_s2_period_returns,
    metrics_from_returns,
)

SLEEVE = "event"
IS_END = pd.Timestamp(RESEARCH_IS_END_S3)
DATA_DIR = s3_data_dir(ROOT)
ARTIFACTS = os.path.join(ROOT, "04_backtest", "s3_trend", "artifacts")
os.makedirs(ARTIFACTS, exist_ok=True)
STACK_PATH = star_stack_path(SLEEVE)
stack = load_star_stack(STACK_PATH)
N_STACK = n_trials_ledger_total(sleeve=SLEEVE)
print("BANNER: IS-only diagnosis — sealed OOS is off-limits")
print(STACK_PATH)
print(f"IS_END={IS_END.date()}  n_trials_stack={N_STACK}")
display(pd.Series(stack, name="value").to_frame())


## 1. Contract


In [ ]:
print("Timing: signal at close t → fill open t+1 (S3 contract)")
print("Costs: A_FX_OANDA_S3 spread + slippage + swap (unless COST_PROFILE_STAR set)")
print("Day boundary: 17:00 America/New_York (pre-registered)")
print("Disposition from this notebook: kill / paper / new H — never re-tune on sealed OOS")


## 2. Run frozen STAR on IS only


In [ ]:
path = price_panel_path("1d", data_dir=DATA_DIR)
if not os.path.isfile(path):
    raise FileNotFoundError(path)
panel = pd.read_parquet(path)
panel["date"] = pd.to_datetime(panel["date"])
panel_is = panel.loc[panel["date"] <= IS_END].copy()
assert (panel_is["date"] > IS_END).sum() == 0, "OOS rows leaked into IS panel"

rates_path = os.path.join(DATA_DIR, "g10_policy_rates.parquet")
rates = pd.read_parquet(rates_path) if os.path.isfile(rates_path) else pd.DataFrame()
reer_path = os.path.join(DATA_DIR, "bis_reer_monthly.parquet")
reer = pd.read_parquet(reer_path) if os.path.isfile(reer_path) else pd.DataFrame()
if not rates.empty and "date" in rates.columns:
    rates = rates.loc[pd.to_datetime(rates["date"]) <= IS_END].copy()
if not reer.empty and "date" in reer.columns:
    reer = reer.loc[pd.to_datetime(reer["date"]) <= IS_END].copy()

cfg = config_from_stack(stack)
s1_weekly = load_s1_period_returns()
s2_daily = load_s2_period_returns()
if s1_weekly.empty:
    warnings.warn("S1 period returns missing — corr_to_s1 will be NaN")
if s2_daily.empty:
    warnings.warn("S2 period returns missing — corr_to_s2 will be NaN")

spy = spy_daily_returns(panel_is["date"].min(), panel_is["date"].max())
res = run_s3_backtest(
    panel_is,
    cfg,
    rates_df=rates if not rates.empty else None,
    reer_df=reer if not reer.empty else None,
    s1_weekly=s1_weekly if not s1_weekly.empty else None,
    s2_daily=s2_daily if not s2_daily.empty else None,
    n_trials_local=1,
    n_trials_stack=N_STACK,
)
is_net = res.returns
is_gross = res.returns_gross
print("IS bars", len(panel_is), "return days", len(is_net))
print(cfg)


## 3. Headline IS metrics (gross vs net)


In [ ]:
headline = oos_headline_metrics(
    is_net,
    is_gross,
    trades=res.pair_trades,
    s1_weekly=s1_weekly if not s1_weekly.empty else None,
    s2_daily=s2_daily if not s2_daily.empty else None,
    spy_daily=spy if not spy.empty else None,
    n_trials_local=1,
    n_trials_stack=N_STACK,
)
display(headline.to_frame("IS"))
fig = plot_equity_net_gross(is_net, is_gross, title="IS equity — net vs gross")
plt.show()
fig = plot_drawdown(is_net, title="IS drawdown (net)")
plt.show()


## 4. Fold-val consistency + early/late IS decay


In [ ]:
is_dates = pd.DatetimeIndex(panel_is["date"].drop_duplicates().sort_values())
folds = make_s3_folds(
    is_dates,
    n_folds=3,
    embargo_bars=embargo_bars_for_config(bar=str(cfg.bar)),
)
fold_df = fold_val_metrics(
    panel_is,
    folds,
    {"frozen_star": cfg},
    hyp_id=None,  # reporting only — stack DSR already set on full-IS run
    sleeve=SLEEVE,
    rates_df=rates if not rates.empty else None,
    reer_df=reer if not reer.empty else None,
    s1_weekly=s1_weekly if not s1_weekly.empty else None,
    s2_daily=s2_daily if not s2_daily.empty else None,
)
display(fold_df)
decay = early_late_is_decay(is_net)
display(decay.to_frame("value"))


## 5. Pair scorecard + attribution (IS)


In [ ]:
score = pair_scorecard(res.book, date_max=IS_END)
attr = pair_pnl_attribution(res.book, date_max=IS_END)
display(score.sort_values("ann_sharpe_net", ascending=False))
display(attr.to_frame("is_total_return"))
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(list(attr.index), 100.0 * attr.values)
ax.set_title("IS pair attribution (% total return)")
ax.set_xlabel("%")
plt.tight_layout()
plt.show()


## 6. Utilization + friction


In [ ]:
util = utilization_stats(is_net, res.pair_trades, weights=res.book.weights)
display(util.to_frame("value"))


## 7. Cost stress (IS)


In [ ]:
stress = cost_stress_sharpes(
    panel_is,
    cfg,
    multipliers=(1.0, 1.5, 2.0),
    rates_df=rates if not rates.empty else None,
    reer_df=reer if not reer.empty else None,
    s1_weekly=s1_weekly if not s1_weekly.empty else None,
    s2_daily=s2_daily if not s2_daily.empty else None,
    n_trials_local=1,
    n_trials_stack=N_STACK,
)
display(stress)


## 8. Regimes (IS)


In [ ]:
regimes = daily_regime_performance(is_net, spy, is_dates)
display(regimes["vol_cutpoints"].to_frame("value"))
display(regimes["market"])
display(regimes["vol"])


## 9. Baselines (raw TSMOM + EW long G10)


In [ ]:
baselines = baseline_configs_from_star(cfg)
raw_res = run_s3_backtest(
    panel_is,
    baselines["raw_tsmom"],
    rates_df=rates if not rates.empty else None,
    reer_df=reer if not reer.empty else None,
    n_trials_local=1,
    n_trials_stack=N_STACK,
)
ew = ew_long_g10_returns(panel_is)
ew = ew.reindex(is_net.index).dropna()
rows = {
    "frozen_star": metrics_from_returns(is_net),
    "raw_tsmom": metrics_from_returns(raw_res.returns),
    "ew_long_g10": metrics_from_returns(ew),
}
base_df = pd.DataFrame(rows).T
display(base_df)


## 10. Fill-timing assert


In [ ]:
timing = check_fill_timing(res.pair_trades, panel_is)
display(timing.head())
print("all_ok rate", float(timing["all_ok"].mean()) if not timing.empty else float("nan"))
if not timing.empty and not bool(timing["all_ok"].all()):
    warnings.warn("Some trades failed fill-timing checks — inspect before trusting PnL")


## 11. Cause-class checklist + post-mortem stub

Paste into `02_research/s3_trend/s3_core_hypothesis_log.md` after filling blanks.


In [ ]:
stub = f"""
### Post-mortem — event STAR IS diagnosis

- Date: {pd.Timestamp.utcnow().strftime('%Y-%m-%d')}
- IS net Sharpe: {headline.get('ann_sharpe_net', float('nan')):.3f}
- IS gross Sharpe: {headline.get('ann_sharpe_gross', float('nan')):.3f}
- Cost bps/year: {headline.get('cost_bps_year', float('nan')):.1f}
- Early/late decay: {decay.get('decay_ratio', float('nan')):.3f}
- Cost stress 2.0× Sharpe: {float(stress.loc[stress['cost_multiplier']==2.0, 'ann_sharpe_net'].iloc[0]) if (stress['cost_multiplier']==2.0).any() else float('nan'):.3f}

**Cause class (pick one):** cost / overfit (fold vs full-IS) / regime / bug / no edge

**Evidence:**
- 

**Disposition:** kill / paper / new H-id (do not re-test sealed OOS after recipe changes)
"""
print(stub)
